# Migrate survey rows from CSV

This notebook updates existing `surveys` rows using the composite key `survey_id` + `respondent_id`.

- The default is a dry run. Set `DRY_RUN = False` only after reviewing the summary.
- CSV rows with no matching database row are skipped and reported in `unmatched_df`.
- Duplicate database matches are skipped and reported in `ambiguous_df`.
- `INSERT_UNMATCHED` is available as an explicit opt-in, but stays disabled by default.
- The full source CSV row is saved in `raw_row_data`; direct survey fields are updated from the mapped source columns.
- Rows are processed in parallel chunks with one database session per worker; progress, commits, failures, elapsed time, and throughput are logged.
- Files named `ext_{cls|ecls}_data_{date}_{bu_name}_{timestamp}.csv` are routed to `{bu_name}_{cls|ecls}_0821`; the `kvn` file prefix is routed to `kvnl`.


In [13]:
from pathlib import Path
from datetime import datetime, timezone
import json
import logging
import os
import re
import sys
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.orm import sessionmaker

# All matching CSV files in this folder are processed one at a time.
INPUT_DIR = Path("/Users/jimmytse/Documents/CLSense-Backend/notebooks/docs/input")
DATABASE_SUFFIX = "0821"
SURVEY_ORDER_CUTOFF = datetime(2026, 7, 31, 23, 59, 59, tzinfo=timezone.utc)  # UTC+0
SURVEY_ORDER_CUTOFF_LABEL = pd.Timestamp(SURVEY_ORDER_CUTOFF).date().isoformat()
FILTERED_OUTPUT_DIR = INPUT_DIR
DRY_RUN = False
INSERT_UNMATCHED = False
BATCH_SIZE = 500
MAX_WORKERS = max(1, int(os.getenv("MIGRATION_MAX_WORKERS", "8")))

# The notebook can be opened from the repository root or from the notebooks directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
BACKEND_ROOT = PROJECT_ROOT / "backend"
sys.path.insert(0, str(BACKEND_ROOT))

from models.Survey import Survey
from utils.database import Base
from config import DATABASE_HOST, DATABASE_PASSWORD, DATABASE_PORT, DATABASE_USER

logger = logging.getLogger("survey_csv_migration")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(
        logging.Formatter("%(asctime)s %(levelname)s [%(threadName)s] %(message)s")
    )
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False
FILENAME_PATTERN = re.compile(
    r"^ext_(?P<survey_type>ecls|cls)_data_(?P<data_date>\d{6})_(?P<bu_name>.+)_(?P<timestamp>\d{8}_\d{6})\.csv$",
    re.IGNORECASE,
)
BU_NAME_ALIASES = {"kvn": "kvnl"}


def parse_input_file(path):
    match = FILENAME_PATTERN.fullmatch(path.name)
    if not match:
        return None
    metadata = match.groupdict()
    metadata["survey_type"] = metadata["survey_type"].lower()
    metadata["bu_name"] = metadata["bu_name"].lower()
    database_bu_name = BU_NAME_ALIASES.get(metadata["bu_name"], metadata["bu_name"])
    metadata["database_name"] = f"{database_bu_name}_{metadata['survey_type']}_{DATABASE_SUFFIX}"
    metadata["path"] = path
    return metadata


def create_database_session_factory(database_name):
    database_uri = (
        f"postgresql://{DATABASE_USER}:{DATABASE_PASSWORD}"
        f"@{DATABASE_HOST}:{DATABASE_PORT}/{database_name}"
    )
    engine = create_engine(
        database_uri,
        pool_pre_ping=True,
        pool_size=MAX_WORKERS,
        max_overflow=0,
        connect_args={"options": "-c timezone=UTC"},
    )
    return engine, sessionmaker(autocommit=False, autoflush=False, bind=engine)


def bootstrap_database_schema_if_empty(database_name):
    engine, _ = create_database_session_factory(database_name)
    try:
        if inspect(engine).has_table("surveys"):
            return False

        logger.warning("Database %s has no surveys table; bootstrapping the project schema", database_name)
        # The current model already contains cls. Remove it temporarily so the
        # historical CSL -> CLS migrations can run normally below.
        Base.metadata.create_all(bind=engine)
        with engine.begin() as connection:
            connection.execute(text('DROP INDEX IF EXISTS "idx_survey_cls"'))
            connection.execute(text('ALTER TABLE "surveys" DROP COLUMN IF EXISTS "cls"'))
        logger.info("Bootstrapped empty database %s", database_name)
        return True
    finally:
        engine.dispose()


KNOWN_MIGRATION_REVISIONS = {
    "0001_baseline",
    "0002_remove_timezone_offsets",
    "0003_restore_timezone_offsets",
    "0004_add_csl",
    "0005_rename_csl_to_cls",
}
HEAD_MIGRATION_REVISION = "0005_rename_csl_to_cls"


def reconcile_alembic_history(database_name, bootstrapped):
    engine, _ = create_database_session_factory(database_name)
    try:
        inspector = inspect(engine)
        current_revisions = []
        if inspector.has_table("alembic_version"):
            with engine.connect() as connection:
                current_revisions = [
                    row[0]
                    for row in connection.execute(text("SELECT version_num FROM alembic_version")).all()
                ]

        if not bootstrapped and current_revisions and all(
            revision in KNOWN_MIGRATION_REVISIONS for revision in current_revisions
        ):
            return

        columns = set()
        if inspector.has_table("surveys"):
            columns = {column["name"] for column in inspector.get_columns("surveys")}
        if "cls" in columns and "csl" not in columns:
            desired_revision = HEAD_MIGRATION_REVISION
        elif "csl" in columns and "cls" not in columns:
            desired_revision = "0004_add_csl"
        else:
            desired_revision = "0003_restore_timezone_offsets"

        if current_revisions:
            logger.warning(
                "Database %s has unknown Alembic revision(s) %s; reconciling to %s based on surveys columns %s",
                database_name,
                current_revisions,
                desired_revision,
                sorted(columns),
            )
        else:
            logger.info("Initializing Alembic history for %s at %s", database_name, desired_revision)

        with engine.begin() as connection:
            connection.execute(
                text("""
                    CREATE TABLE IF NOT EXISTS alembic_version (
                        version_num VARCHAR(32) NOT NULL PRIMARY KEY
                    )
                """)
            )
            connection.execute(text("DELETE FROM alembic_version"))
            connection.execute(
                text("INSERT INTO alembic_version (version_num) VALUES (:revision)"),
                {"revision": desired_revision},
            )
    finally:
        engine.dispose()


def run_alembic_command(database_name, *arguments):
    migration_env = os.environ.copy()
    migration_env.update(
        {
            "DATABASE_USER": str(DATABASE_USER),
            "DATABASE_PASSWORD": str(DATABASE_PASSWORD),
            "DATABASE_HOST": str(DATABASE_HOST),
            "DATABASE_PORT": str(DATABASE_PORT),
        }
    )
    migration_env["DATABASE_NAME"] = database_name
    migration_env["PYTHONPATH"] = os.pathsep.join(
        [str(BACKEND_ROOT), migration_env.get("PYTHONPATH", "")]
    ).rstrip(os.pathsep)
    command = [
        sys.executable,
        "-m",
        "alembic",
        "-c",
        str(BACKEND_ROOT / "alembic.ini"),
    ]
    command.extend(arguments)
    logger.info("Running Alembic for %s: %s", database_name, " ".join(arguments))
    completed = subprocess.run(
        command,
        cwd=BACKEND_ROOT,
        env=migration_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.stdout:
        for line in completed.stdout.rstrip().splitlines():
            logger.info("Alembic [%s]: %s", database_name, line)
    if completed.returncode != 0:
        raise RuntimeError(
            f"Alembic command failed for {database_name} with exit code {completed.returncode}.\n"
            f"Alembic output:\n{completed.stdout or '<no output>'}"
        )


def run_database_migrations(database_name):
    bootstrapped = bootstrap_database_schema_if_empty(database_name)
    reconcile_alembic_history(database_name, bootstrapped)
    run_alembic_command(database_name, "upgrade", "head")
    logger.info("Database migrations finished for %s", database_name)


input_files = []
unrecognized_files = []
for input_path in sorted(INPUT_DIR.glob("*.csv")):
    metadata = parse_input_file(input_path)
    if metadata is None:
        unrecognized_files.append(input_path)
    else:
        input_files.append(metadata)
if not input_files:
    raise FileNotFoundError(f"No matching CSV files found in {INPUT_DIR}")
if unrecognized_files:
    logger.warning("Ignoring %s files with names outside the expected convention", len(unrecognized_files))

print(f"Project root: {PROJECT_ROOT}")
print(f"Input folder: {INPUT_DIR}")
print(f"Matching CSV files: {len(input_files)}")
print(f"Target databases: {sorted({item['database_name'] for item in input_files})}")
print(f"Dry run: {DRY_RUN}")
print(f"Insert unmatched: {INSERT_UNMATCHED}")
print(f"Survey order date cutoff: {SURVEY_ORDER_CUTOFF_LABEL}")
print(f"Filtered CSV output folder: {FILTERED_OUTPUT_DIR}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Worker threads: {MAX_WORKERS}")

Project root: /Users/jimmytse/Documents/CLSense-Backend
CSV: /Users/jimmytse/Documents/CLSense-Backend/notebooks/docs/input/ext_cls_data_202607_wtchk_20260819_131435.csv
Dry run: False
Insert unmatched: False
Survey order date cutoff: 2026-07-31
Filtered CSV output: /Users/jimmytse/Documents/CLSense-Backend/notebooks/docs/input/ext_cls_data_202607_wtchk_20260819_131435-survey-order-after-2026-07-31.csv
Batch size: 500
Worker threads: 8


In [14]:
def normalize_key(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    # Remove only a pandas-style trailing .0; preserve meaningful leading zeroes.
    if text.endswith(".0") and text[:-2].isdigit():
        return text[:-2]
    return text


def nullable_float(value):
    if value is None or pd.isna(value) or str(value).strip() == "":
        return None
    return float(value)


def nullable_int(value):
    parsed = nullable_float(value)
    return None if parsed is None else int(parsed)


def nullable_datetime(value):
    if value is None or pd.isna(value) or str(value).strip() == "":
        return None
    parsed = pd.to_datetime(value, utc=True, errors="coerce")
    if pd.isna(parsed):
        raise ValueError(f"Invalid datetime: {value!r}")
    return parsed.to_pydatetime()


def nullable_bool_from_delete_flag(value):
    if value is None or pd.isna(value) or str(value).strip() == "":
        return False
    return str(value).strip().upper() == "Y"


def source_value(row, *column_names):
    for column_name in column_names:
        if column_name in row.index:
            value = row[column_name]
            if isinstance(value, pd.Series):
                return value.iloc[0]
            return value
    return None

In [15]:
def read_csv_for_migration(csv_path, filtered_csv_path):
    # Read IDs as strings so leading zeroes and large identifiers are not changed.
    df = pd.read_csv(
        csv_path,
        dtype=str,
        keep_default_na=False,
        na_values=[""],
        encoding="utf-8",
    )

    required_columns = {"survey_id", "respondent_id"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"CSV is missing required columns: {sorted(missing_columns)}")
    if "survey_order_date" not in df.columns:
        raise ValueError("CSV is missing required column: survey_order_date")

    source_survey_order_dates = df["survey_order_date"]
    if isinstance(source_survey_order_dates, pd.DataFrame):
        logger.warning("%s has duplicate survey_order_date columns; using the first one", csv_path.name)
        source_survey_order_dates = source_survey_order_dates.iloc[:, 0]
    parsed_survey_order_dates = pd.to_datetime(source_survey_order_dates, utc=True, errors="coerce")
    cutoff_timestamp = pd.Timestamp(SURVEY_ORDER_CUTOFF)
    if cutoff_timestamp.tzinfo is None:
        cutoff_timestamp = cutoff_timestamp.tz_localize("UTC")
    else:
        cutoff_timestamp = cutoff_timestamp.tz_convert("UTC")
    cutoff_exclusive = cutoff_timestamp.normalize() + pd.Timedelta(days=1)
    recent_row_mask = parsed_survey_order_dates >= cutoff_exclusive
    recent_rows_df = df.loc[recent_row_mask].copy()
    filtered_csv_path.parent.mkdir(parents=True, exist_ok=True)
    recent_rows_df.to_csv(filtered_csv_path, index=False, encoding="utf-8")
    logger.info("%s: saved %s rows after %s to %s", csv_path.name, len(recent_rows_df), cutoff_timestamp.date().isoformat(), filtered_csv_path)

    df["_survey_key"] = df["survey_id"].map(normalize_key)
    df["_respondent_key"] = df["respondent_id"].map(normalize_key)
    invalid_key_rows = df[df["_survey_key"].isna() | df["_respondent_key"].isna()].copy()
    work_df = df[df["_survey_key"].notna() & df["_respondent_key"].notna()].copy()
    logger.info("%s: read=%s, usable_keys=%s, missing_keys=%s", csv_path.name, len(df), len(work_df), len(invalid_key_rows))
    return df, recent_rows_df, invalid_key_rows, work_df

2026-08-21 14:16:14,782 INFO [MainThread] Saved 2153 rows with survey_order_date after 2026-07-31 to /Users/jimmytse/Documents/CLSense-Backend/notebooks/docs/input/ext_cls_data_202607_wtchk_20260819_131435-survey-order-after-2026-07-31.csv


Read 77,541 CSV rows
Rows with usable composite keys: 77,541
Rows with missing composite keys: 0
Rows saved to filtered CSV: 2,153


,survey_type,survey_id,respondent_id,bu_key,store_key,question_id,submitdate,survey_order_date,reporting_month,etl_last_update,answer,CLS,is_delete
0,cls_long_survey,588539,207098,WTCHK,3389,Q6Comment,2026-08-16T23:39:34.000Z,2026-08-10T12:36:21.000Z,202608,2026-08-16T19:13:11.403Z,阿秀服務好,100.0,N
1,cls_long_survey,588539,207096,WTCHK,3229,Q6Comment,2026-08-16T23:36:36.000Z,2026-08-11T18:26:01.000Z,202608,2026-08-16T19:13:11.403Z,員工專業又細心,100.0,N
2,cls_long_survey,588539,207091,WTCHK,3551,Q6Comment,2026-08-16T23:29:13.000Z,2026-08-08T21:42:06.000Z,202608,2026-08-16T19:13:11.403Z,敏，很好，很细心,100.0,N
3,cls_long_survey,588539,207086,WTCHK,3632,Q6Comment,2026-08-16T23:24:48.000Z,2026-08-08T18:00:03.000Z,202608,2026-08-16T19:13:11.403Z,店員Maggie收銀效率快，好尃業，提我換購優惠產品,100.0,N
4,cls_long_survey,588539,207085,WTCHK,3688,Q6Comment,2026-08-16T23:19:23.000Z,2026-08-14T19:56:06.000Z,202608,2026-08-16T19:13:11.403Z,Carmen good Service,100.0,N


,survey_type,survey_id,respondent_id,bu_key,store_key,question_id,submitdate,survey_order_date,reporting_month,etl_last_update,answer,CLS,is_delete,_survey_key,_respondent_key
0,cls_long_survey,588539,207098,WTCHK,3389,Q6Comment,2026-08-16T23:39:34.000Z,2026-08-10T12:36:21.000Z,202608,2026-08-16T19:13:11.403Z,阿秀服務好,100.0,N,588539,207098
1,cls_long_survey,588539,207096,WTCHK,3229,Q6Comment,2026-08-16T23:36:36.000Z,2026-08-11T18:26:01.000Z,202608,2026-08-16T19:13:11.403Z,員工專業又細心,100.0,N,588539,207096
2,cls_long_survey,588539,207091,WTCHK,3551,Q6Comment,2026-08-16T23:29:13.000Z,2026-08-08T21:42:06.000Z,202608,2026-08-16T19:13:11.403Z,敏，很好，很细心,100.0,N,588539,207091
3,cls_long_survey,588539,207086,WTCHK,3632,Q6Comment,2026-08-16T23:24:48.000Z,2026-08-08T18:00:03.000Z,202608,2026-08-16T19:13:11.403Z,店員Maggie收銀效率快，好尃業，提我換購優惠產品,100.0,N,588539,207086
4,cls_long_survey,588539,207085,WTCHK,3688,Q6Comment,2026-08-16T23:19:23.000Z,2026-08-14T19:56:06.000Z,202608,2026-08-16T19:13:11.403Z,Carmen good Service,100.0,N,588539,207085


## Field mapping

The CSV contains more fields than the `surveys` table. The mapped fields below are updated directly. `survey_type`, `question_id`, `reporting_month`, `bu_key`, and other source-only fields remain available inside `raw_row_data`. Derived sentiment, topic, keyword, and department relationships are intentionally left unchanged.

In [16]:
def build_update_values(row, row_index):
    # CLS is the canonical score field; accept CSL for legacy extracts.
    cls_raw = source_value(row, "CLS", "CSL", "cls", "csl")
    values = {
        "store_key": nullable_int(source_value(row, "store_key")),
        "comment": source_value(row, "answer", "comment"),
        "reported_at": nullable_datetime(source_value(row, "survey_order_date", "reported_at")),
        "updated_at": nullable_datetime(source_value(row, "etl_last_update", "updated_at")),
        "is_deleted": nullable_bool_from_delete_flag(source_value(row, "is_delete", "is_deleted")),
        "cls": nullable_float(cls_raw),
        "raw_row_data": json.dumps(
            {
                "index": row_index,
                "row": {
                    str(key): value
                    for key, value in row.to_dict().items()
                    if not str(key).startswith("_")
                },
            },
            default=str,
        ),
    }
    # Do not overwrite a field from a source column that is not present.
    if "store_key" not in row.index:
        values.pop("store_key")
    if not ({"answer", "comment"} & set(row.index)):
        values.pop("comment")
    if not ({"survey_order_date", "reported_at"} & set(row.index)):
        values.pop("reported_at")
    if not ({"etl_last_update", "updated_at"} & set(row.index)):
        values.pop("updated_at")
    if not ({"is_delete", "is_deleted"} & set(row.index)):
        values.pop("is_deleted")
    if not ({"CLS", "CSL", "cls", "csl"} & set(row.index)):
        values.pop("cls")
    return values

In [17]:
def chunked(items, chunk_size):
    for start in range(0, len(items), chunk_size):
        yield items[start:start + chunk_size]


def process_chunk(chunk_number, chunk, session_factory):
    # A SQLAlchemy Session is never shared between worker threads.
    db = session_factory()
    results = []
    update_mappings = []
    insert_objects = []
    try:
        logger.info("Worker started chunk %s with %s rows", chunk_number, len(chunk))
        for row_index, row in chunk:
            survey_id = row["_survey_key"]
            respondent_id = row["_respondent_key"]
            csv_row = row_index + 2
            base_result = {"csv_row": csv_row, "survey_id": survey_id, "respondent_id": respondent_id}
            try:
                matches = (
                    db.query(Survey.id)
                    .filter(
                        Survey.survey_id == survey_id,
                        Survey.respondent_id == respondent_id,
                    )
                    .all()
                )

                if len(matches) > 1:
                    results.append({**base_result, "status": "ambiguous", "matches": len(matches)})
                    logger.warning("Chunk %s row %s skipped: %s database matches for key (%s, %s)", chunk_number, csv_row, len(matches), survey_id, respondent_id)
                    continue

                values = build_update_values(row, row_index)
                if not matches:
                    result = {**base_result, "status": "unmatched"}
                    if not INSERT_UNMATCHED:
                        results.append(result)
                        continue
                    if values.get("store_key") is None:
                        results.append({**base_result, "status": "failed", "error": "store_key is required for insert"})
                        continue
                    insert_objects.append(Survey(survey_id=survey_id, respondent_id=respondent_id, **values))
                    result["status"] = "inserted"
                    results.append(result)
                else:
                    database_id = matches[0][0]
                    update_mappings.append({"id": database_id, **values})
                    results.append({**base_result, "status": "updated", "database_id": database_id})
            except Exception as error:
                db.rollback()
                logger.exception("Chunk %s row %s failed for key (%s, %s)", chunk_number, csv_row, survey_id, respondent_id)
                results.append({**base_result, "status": "failed", "error": str(error)})

        write_count = len(update_mappings) + len(insert_objects)
        if DRY_RUN:
            db.rollback()
            logger.info("Worker completed chunk %s in dry-run mode: %s candidate writes", chunk_number, write_count)
        else:
            if update_mappings:
                db.bulk_update_mappings(Survey, update_mappings)
            if insert_objects:
                db.add_all(insert_objects)
            if write_count:
                db.commit()
            logger.info("Worker committed chunk %s: %s updates/inserts", chunk_number, write_count)
    except Exception as error:
        db.rollback()
        logger.exception("Chunk %s rolled back: %s", chunk_number, error)
        for result in results:
            if result["status"] in {"updated", "inserted"}:
                result["status"] = "failed"
                result["error"] = f"Chunk rolled back: {error}"
    finally:
        db.close()
    return results


def migrate_one_file(file_metadata):
    csv_path = file_metadata["path"]
    database_name = file_metadata["database_name"]
    filtered_csv_path = FILTERED_OUTPUT_DIR / f"{csv_path.stem}-survey-order-after-{SURVEY_ORDER_CUTOFF_LABEL}.csv"
    logger.info("Starting %s -> database %s", csv_path.name, database_name)
    _, recent_rows_df, invalid_key_rows, work_df = read_csv_for_migration(csv_path, filtered_csv_path)
    engine, session_factory = create_database_session_factory(database_name)
    started_at = time.monotonic()
    try:
        work_items = [(row_index, row) for row_index, row in work_df.iterrows()]
        chunks = list(chunked(work_items, BATCH_SIZE))
        worker_count = min(MAX_WORKERS, max(1, len(chunks)))
        file_results = []
        logger.info("%s: %s rows, %s chunks, %s worker threads, dry_run=%s", csv_path.name, len(work_items), len(chunks), worker_count, DRY_RUN)

        with ThreadPoolExecutor(max_workers=worker_count, thread_name_prefix=f"survey-{file_metadata['bu_name']}-{file_metadata['survey_type']}") as executor:
            futures = [
                executor.submit(process_chunk, chunk_number, chunk, session_factory)
                for chunk_number, chunk in enumerate(chunks, start=1)
            ]
            for completed_chunks, future in enumerate(as_completed(futures), start=1):
                chunk_results = future.result()
                for result in chunk_results:
                    result["source_file"] = csv_path.name
                    result["database_name"] = database_name
                file_results.extend(chunk_results)
                processed = min(completed_chunks * BATCH_SIZE, len(work_items))
                elapsed = time.monotonic() - started_at
                rate = processed / elapsed if elapsed else 0
                logger.info("%s progress: %s/%s rows (%.1f%%), %.1f rows/sec", csv_path.name, processed, len(work_items), (processed / len(work_items) * 100) if work_items else 100, rate)

        summary = {
            "source_file": csv_path.name,
            "database_name": database_name,
            "rows_read": len(work_df) + len(invalid_key_rows),
            "rows_saved_after_cutoff": len(recent_rows_df),
            "updated": sum(result["status"] == "updated" for result in file_results),
            "unmatched": sum(result["status"] == "unmatched" for result in file_results),
            "ambiguous": sum(result["status"] == "ambiguous" for result in file_results),
            "inserted": sum(result["status"] == "inserted" for result in file_results),
            "failed": sum(result["status"] == "failed" for result in file_results),
            "elapsed_seconds": time.monotonic() - started_at,
        }
        logger.info("Finished %s -> %s: updated=%s, unmatched=%s, failed=%s", csv_path.name, database_name, summary["updated"], summary["unmatched"], summary["failed"])
        return summary, file_results
    finally:
        engine.dispose()

target_databases = sorted({item["database_name"] for item in input_files})
for database_name in target_databases:
    run_database_migrations(database_name)

migration_summaries = []
all_results = []
for file_metadata in input_files:
    try:
        summary, file_results = migrate_one_file(file_metadata)
    except Exception as error:
        logger.exception("File failed %s -> %s; continuing with remaining files", file_metadata["path"].name, file_metadata["database_name"])
        summary = {
            "source_file": file_metadata["path"].name,
            "database_name": file_metadata["database_name"],
            "status": "failed",
            "error": str(error),
        }
        file_results = []
    migration_summaries.append(summary)
    all_results.extend(file_results)

updated_rows = [result for result in all_results if result["status"] == "updated"]
unmatched_rows = [result for result in all_results if result["status"] == "unmatched"]
ambiguous_rows = [result for result in all_results if result["status"] == "ambiguous"]
inserted_rows = [result for result in all_results if result["status"] == "inserted"]
failed_rows = [result for result in all_results if result["status"] == "failed"]
display(pd.DataFrame(migration_summaries))
logger.info("All files finished: files=%s, updated=%s, unmatched=%s, ambiguous=%s, inserted=%s, failed=%s", len(migration_summaries), len(updated_rows), len(unmatched_rows), len(ambiguous_rows), len(inserted_rows), len(failed_rows))

2026-08-21 14:16:24,035 INFO [MainThread] Starting migration: 77541 rows, 156 chunks, 8 worker threads, dry_run=False
2026-08-21 14:16:24,035 INFO [survey-migration_0] Worker started chunk 1 with 500 rows
2026-08-21 14:16:24,035 INFO [survey-migration_1] Worker started chunk 2 with 500 rows
2026-08-21 14:16:24,035 INFO [survey-migration_2] Worker started chunk 3 with 500 rows
2026-08-21 14:16:24,036 INFO [survey-migration_3] Worker started chunk 4 with 500 rows
2026-08-21 14:16:24,036 INFO [survey-migration_4] Worker started chunk 5 with 500 rows
2026-08-21 14:16:24,036 INFO [survey-migration_5] Worker started chunk 6 with 500 rows
2026-08-21 14:16:24,036 INFO [survey-migration_6] Worker started chunk 7 with 500 rows
2026-08-21 14:16:24,036 INFO [survey-migration_7] Worker started chunk 8 with 500 rows
2026-08-21 14:16:30,212 INFO [survey-migration_0] Worker committed chunk 1: 0 updates/inserts
2026-08-21 14:16:30,215 INFO [survey-migration_1] Worker committed chunk 2: 0 updates/insert

Matched rows updated: 61,200
Unmatched rows: 16,341
Ambiguous rows skipped: 0
Rows inserted: 0
Rows failed: 0


In [ ]:
unmatched_df = pd.DataFrame(unmatched_rows)
ambiguous_df = pd.DataFrame(ambiguous_rows)
failed_df = pd.DataFrame(failed_rows)
updated_df = pd.DataFrame(updated_rows)
inserted_df = pd.DataFrame(inserted_rows)

# Review these before changing DRY_RUN to False.
display(unmatched_df.head(20))
display(ambiguous_df.head(20))
display(failed_df.head(20))

,csv_row,survey_id,respondent_id,status
0,2515,646344,1288967,unmatched
1,2523,646344,1288946,unmatched
2,2527,646344,1288937,unmatched
3,2533,646344,1288921,unmatched
4,2535,646344,1288916,unmatched
5,2536,646344,1288914,unmatched
6,2538,646344,1288913,unmatched
7,2540,646344,1288910,unmatched
8,2541,646344,1288908,unmatched
9,2544,646344,1288894,unmatched


""


""


## Commit checklist

1. Confirm the CSV path and database environment.
2. Run with `DRY_RUN = True` and review `updated_df`, `unmatched_df`, `ambiguous_df`, and `failed_df`.
3. If the unmatched rows should be created, set `INSERT_UNMATCHED = True` and confirm each row has a valid existing `store_key`.
4. Set `DRY_RUN = False` to commit the update in batches.